# Day 4 Assignments

This notebook contains all four data-analysis assignments in one file.
Run each section individually to view the output.

In [ ]:
from pathlib import Path
import pandas as pd

base_dir = Path.cwd()
required_files = [
    'patient_clinical_data_raw.csv',
    'ecommerce_orders_raw.csv',
    'iot_sensor_data_raw.csv',
    'customers.csv',
    'products.csv',
    'orders.csv'
]
if not all((base_dir / name).exists() for name in required_files):
    base_dir = base_dir.parent

base_dir

## Assignment 2: E-Commerce Orders

Explore order data, clean invalid values, and compute revenue insights.

In [ ]:
orders = pd.read_csv(base_dir / 'ecommerce_orders_raw.csv')
print('Shape:', orders.shape)
print('\nMissing values:')
print(orders.isnull().sum())

orders['Rating'] = pd.to_numeric(orders['Rating'], errors='coerce')
orders['Rating'] = orders['Rating'].fillna(orders['Rating'].median())
orders['City'] = orders['City'].str.title().str.strip()
orders['Quantity'] = pd.to_numeric(orders['Quantity'], errors='coerce')
orders = orders[orders['Quantity'] > 0].drop_duplicates()
orders['Order_Date'] = pd.to_datetime(orders['Order_Date'])

offers = orders.copy()
offers['Gross_Amount'] = offers['Quantity'] * offers['Unit_Price']
offers['Discount_Amount'] = offers['Gross_Amount'] * offers['Discount'] / 100
offers['Net_Amount'] = offers['Gross_Amount'] - offers['Discount_Amount']

offers['Rating_Category'] = pd.cut(
    offers['Rating'],
    bins=[0, 2, 3, 4, 5],
    labels=['Poor', 'Average', 'Good', 'Excellent'],
    right=False
)
offers['Month'] = offers['Order_Date'].dt.month

print('\nTotal revenue:', round(offers['Net_Amount'].sum(), 2))
print('Average order value:', round(offers['Net_Amount'].mean(), 2))
print('\nRevenue by category:')
print(offers.groupby('Product_Category')['Net_Amount'].sum().sort_values(ascending=False))
print('\nTop 10 customers by spending:')
print(offers.groupby('Customer_ID')['Net_Amount'].sum().sort_values(ascending=False).head(10))
print('\nMonthly revenue:')
print(offers.groupby('Month')['Net_Amount'].sum())